# 04 · Cleaning Data Basics

Real invoice data is never perfectly clean: missing amounts, duplicate rows, inconsistent
text casing. This notebook covers the core cleaning toolkit.


In [1]:
import pandas as pd

df = pd.read_csv("../data/synthetic_ap_invoices.csv", parse_dates=["Invoice Date"])
df.shape

(305, 9)

## Handling missing values

Before deleting anything, decide **why** it's missing. In our dataset, a blank `Hold Reason`
just means the invoice isn't on hold — that's meaningful, not an error. A blank `Invoice Amount`
is a genuine data quality issue.


In [2]:
df.isna().sum()   # missing value count per column, to plan your approach

Invoice Number      0
Vendor Code         0
Region              0
Invoice Date        0
Invoice Amount      6
Currency            0
PO Number           0
Hold Reason       109
Payment Status      0
dtype: int64

In [3]:
# Option 1: fill NaN with a meaningful placeholder (better than deleting, when NaN is meaningful)
df['Hold Reason'] = df['Hold Reason'].fillna('Not on Hold')
df['Hold Reason'].value_counts()

Hold Reason
Not on Hold          109
Missing PO            37
Price Difference      36
Approval Pending      34
Quantity Mismatch     31
Duplicate Invoice     30
Missing GR            28
Name: count, dtype: int64

In [4]:
# Option 2: drop rows where a specific column is missing (when the missing value is an actual error)
df_clean = df.dropna(subset=['Invoice Amount'])
print(df.shape, '->', df_clean.shape)

(305, 9) -> (299, 9)


In [5]:
# Option 3: drop rows only if EVERY column is blank (rare, but useful for junk rows)
df_clean = df.dropna(how='all')

# Option 4: drop columns that are entirely blank
df_clean = df.dropna(axis=1, how='all')

## Finding and removing duplicates

Duplicate invoice numbers are a classic AP data quality issue (double-keyed invoices,
export glitches, etc).


In [6]:
df.duplicated().sum()   # fully duplicate rows across ALL columns

np.int64(5)

In [7]:
df['Invoice Number'].duplicated().sum()   # duplicate on just Invoice Number — the real business key

np.int64(5)

In [8]:
dupes = df[df.duplicated(subset=['Invoice Number'], keep=False)]
dupes.sort_values('Invoice Number').head(10)

,Invoice Number,Vendor Code,Region,Invoice Date,Invoice Amount,Currency,PO Number,Hold Reason,Payment Status
21,FR15-INV-200098,V165725,FR15,2026-06-23,22543.78,GBP,PO1378165,Not on Hold,On Hold
181,FR15-INV-200098,V165725,FR15,2026-06-23,22543.78,GBP,PO1378165,Not on Hold,On Hold
70,IT25-INV-200039,V291335,IT25,2026-07-14,11122.03,GBP,PO4376513,Not on Hold,On Hold
138,IT25-INV-200039,V291335,IT25,2026-07-14,11122.03,GBP,PO4376513,Not on Hold,On Hold
6,UK30-INV-200271,V299041,UK30,2026-03-15,16201.04,GBP,PO7849486,Not on Hold,On Hold
15,UK30-INV-200271,V299041,UK30,2026-03-15,16201.04,GBP,PO7849486,Not on Hold,On Hold
243,UK30-INV-200296,V154886,UK30,2026-04-30,16765.78,EUR,PO5684346,Not on Hold,On Hold
272,UK30-INV-200296,V154886,UK30,2026-04-30,16765.78,EUR,PO5684346,Not on Hold,On Hold
97,US20-INV-200267,V221958,US20,2026-03-08,10876.74,GBP,PO1326916,Approval Pending,On Hold
216,US20-INV-200267,V221958,US20,2026-03-08,10876.74,GBP,PO1326916,Approval Pending,On Hold


In [9]:
# keep='first' keeps the first occurrence and drops the rest; adjust based on your business rule
df_deduped = df.drop_duplicates(subset=['Invoice Number'], keep='first')
print(df.shape, '->', df_deduped.shape)

(305, 9) -> (300, 9)


## Fixing data types

Amounts, dates, and codes often load with the wrong dtype from a messy export.


In [10]:
df.dtypes

Invoice Number               str
Vendor Code                  str
Region                       str
Invoice Date      datetime64[us]
Invoice Amount           float64
Currency                     str
PO Number                    str
Hold Reason                  str
Payment Status               str
dtype: object

In [11]:
df['Vendor Code'] = df['Vendor Code'].astype(str)          # force text, avoid numeric coercion
df['Invoice Amount'] = pd.to_numeric(df['Invoice Amount'], errors='coerce')  # force numeric, bad values -> NaN

## Cleaning text columns

Inconsistent casing/whitespace is extremely common in vendor names, codes, and free-text fields.

```python
df['Vendor Code'] = df['Vendor Code'].str.strip()      # remove leading/trailing whitespace
df['Vendor Code'] = df['Vendor Code'].str.upper()       # standardize casing
```


## Exercise

1. Count how many invoices have a genuinely missing `Invoice Amount`.
2. Deduplicate on `Invoice Number`, keeping the **last** occurrence instead of the first — how does the row count change?
3. Create a new column `Is On Hold` that's `True`/`False` based on whether `Hold Reason` is not null (do this BEFORE filling NaN with a placeholder).


In [12]:
# Your code here
